# Überblick: Logistische Regression mit scikit-learn

## Thema und Ziel
Dieses Notebook führt durch einen typischen Machine-Learning-Ablauf: Aus GMAT-Punktzahl, GPA und Berufserfahrung wird vorhergesagt, ob eine Person an einer Universität angenommen wird. Das Beispiel nutzt kleine, vorbereitete Daten und zeigt die wichtigsten Schritte mit scikit-learn.

## Das lernst du
- Eingabemerkmale `X` und Zielwerte `y` festlegen und in Trainings- und Testdaten teilen.
- Ein Modell mit `fit` trainieren und mit `predict` testen.
- Vorhersagen mit Accuracy und Konfusionsmatrix beurteilen.
- Hyperparameter ansehen und mit einem anderen Wahrscheinlichkeitsschwellenwert experimentieren.

## Wichtige Begriffe
| Begriff | Einfach erklärt |
|---|---|
| Logistische Regression | Ein Klassifikationsmodell, das eine Wahrscheinlichkeit für eine Klasse schätzt. |
| Merkmal und Zielwert | Merkmale sind Eingaben wie `gmat`, `gpa` und `work_experience`; `admitted` ist das vorherzusagende Ergebnis. |
| Trainings- und Testdaten | Trainingsdaten dienen zum Lernen; zurückgehaltene Testdaten prüfen, wie gut das Modell auf unbekannten Fällen arbeitet. |
| Accuracy | Anteil aller Vorhersagen, die richtig sind. Bei ungleichen Klassen kann sie ein unvollständiges Bild geben. |
| Konfusionsmatrix | Tabelle mit richtigen und falschen Vorhersagen für jede Klasse. |
| Hyperparameter | Einstellungen, die vor dem Training festgelegt werden. Bei `LogisticRegression` regelt `C` die Stärke der Regularisierung: kleineres `C` bedeutet stärkere Begrenzung der Modellgewichte. |
| Schwellenwert | Grenzwert für die Umwandlung einer Wahrscheinlichkeit in Klasse 0 oder 1. Ein niedrigerer Grenzwert sagt häufiger Klasse 1 voraus. |

## Zentrale Syntax und benötigte Werte
- `pd.DataFrame(candidates, ...)` baut aus den vorbereiteten Listen eine Tabelle `df`.
- `train_test_split(X, y, test_size=..., random_state=...)` teilt Eingaben und Zielwerte; daraus entstehen `X_train`, `X_test`, `y_train` und `y_test`.
- `lr = LogisticRegression()` erstellt das Modell und `lr.fit(X_train, y_train)` trainiert es.
- `lr.predict(X_test)` erzeugt Klassen; `lr.predict_proba(X_test)[:, 1]` liefert die Wahrscheinlichkeit für Klasse 1.
- `confusion_matrix(y_test, y_pred)` und `accuracy_score(y_test, y_pred)` vergleichen Vorhersagen mit den echten Testwerten.
- `lr.coef_`, `lr.intercept_` und `lr.get_params()` zeigen gelernte Parameter beziehungsweise Einstellungen des Modells.

Code-Zellen führen die einzelnen Schritte aus; Markdown-Zellen erläutern den Ablauf und die Übungsfragen.

# Logistische Regression mit scikit-learn

Dieses Notebook konzentriert sich auf den **workflow** zur Umsetzung einer logistischen Regression mit scikit-learn. Das Beispiel zeigt, wie sich die vorliegende Aufgabe mit logistischer Regression lösen lässt.

Am Ende solltest du die nötigen Schritte zum Trainieren eines Modells mit scikit-learn kennen und es auf unbekannte Daten anwenden können.

## Daten

Nehmen wir als einfaches Beispiel an, du möchtest mit Python ein Modell bauen, das vorhersagt, ob Bewerberinnen und Bewerber an einer angesehenen Universität angenommen werden.

Es gibt zwei mögliche Ergebnisse: angenommen (Wert `1`) oder abgelehnt (Wert `0`).

Die logistische Regression verwendet dabei:
- die Zielvariable, die angibt, ob eine Person angenommen wurde;
- die drei unabhängigen Merkmale GMAT-Punktzahl, GPA und Berufserfahrung in Jahren.

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
# Generating the data
candidates = {
    "gmat": [
        780, 750, 690, 710, 680, 730, 690, 720, 740, 690,
        610, 690, 710, 680, 770, 610, 580, 650, 540, 590,
        620, 600, 550, 550, 570, 670, 660, 580, 650, 660,
        640, 620, 660, 660, 680, 650, 670, 580, 590, 690,
    ],
    "gpa": [
        4, 3.9, 3.3, 3.7, 3.9, 3.7, 2.3, 3.3, 3.3, 1.7,
        2.7, 3.7, 3.7, 3.3, 3.3, 3, 2.7, 3.7, 2.7, 2.3,
        3.3, 2, 2.3, 2.7, 3, 3.3, 3.7, 2.3, 3.7, 3.3,
        3, 2.7, 4, 3.3, 3.3, 2.3, 2.7, 3.3, 1.7, 3.7,
    ],
    "work_experience": [
        3, 4, 3, 5, 4, 6, 1, 4, 5, 1,
        3, 5, 6, 4, 3, 1, 4, 6, 2, 3,
        2, 1, 4, 1, 2, 6, 4, 2, 6, 5,
        1, 2, 4, 6, 5, 1, 2, 1, 4, 5,
    ],
    "admitted": [
        1, 1, 1, 1, 1, 1, 0, 1, 1, 0,
        0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 1, 1,
        0, 0, 1, 1, 1, 0, 0, 0, 0, 1,
    ],
}

df = pd.DataFrame(
    candidates, columns=pd.Index(["gmat", "gpa", "work_experience", "admitted"])
)
df.head()

In [ ]:
# Defining target and predictors
X = df[["gmat", "gpa", "work_experience"]]
y = df["admitted"]

In [ ]:
# Train-test-split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

## Logistische Regression erstellen

In [ ]:
# Modelling
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

### Konfusionsmatrix

In [ ]:
# Confusion matrix using pandas crosstab
conf_matrix = pd.crosstab(y_test, y_pred, rownames=["Actual"], colnames=["Predicted"])
sns.heatmap(conf_matrix, annot=True);

In [ ]:
print("Accuracy: ", accuracy_score(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
# Confusion matrix using the confusion_matrix function from sklearn
cfm = confusion_matrix(y_test, y_pred)
sns.heatmap(cfm, cmap="YlGnBu", annot=True, fmt="d", linewidths=0.5);

**Check your understanding**

- Sieh in der scikit-learn-Dokumentation zur logistischen Regression nach, wie du die geschätzten Parameter deines Modells ausgeben kannst.

<details>
<summary>Show answer</summary>

Nach dem Training stehen die gelernten Parameter im Modellobjekt: `lr.coef_` enthält das Gewicht für jedes Merkmal und `lr.intercept_` den Achsenabschnitt. Das sind die Werte $b_1$ und $b_0$ des logistischen Modells. Du kannst sie direkt ausgeben, zum Beispiel mit `print(lr.coef_, lr.intercept_)`.
</details>

- scikit-learn legt für viele Hyperparameter sinnvolle Standardwerte fest. Gib alle Hyperparameter deines Modells aus und prüfe, was sie bedeuten und welche weiteren Optionen es gibt.

<details>
<summary>Show answer</summary>

Mit `lr.get_params()` siehst du alle Hyperparameter und ihre aktuellen Werte. Wichtige Beispiele für die logistische Regression sind `C` (inverse Stärke der Regularisierung: ein kleinerer Wert bedeutet stärkere Regularisierung), `penalty` (Art der Regularisierung, zum Beispiel `l2`), `solver` (Optimierungsverfahren) und `max_iter` (maximale Zahl der Iterationen). Die scikit-learn-Dokumentation erklärt die erlaubten Optionen.
</details>

- Ändere den Schwellenwert auf 0,3 und gib die Konfusionsmatrix für diesen Wert aus.

<details>
<summary>Click here for a hint for the last task</summary>
Du musst die vorhergesagten Wahrscheinlichkeiten ausgeben und eine Funktion verwenden, die den Wert 0 liefert, wenn die Wahrscheinlichkeit höchstens 0,3 beträgt, und sonst den Wert 1. Verwende dieses Array anschließend als `y_pred`, um die Konfusionsmatrix zu berechnen.
</details>

<details>
<summary>Show answer</summary>

`predict()` verwendet immer einen festen Grenzwert von 0,5. Für einen anderen Wert verwendest du daher die vorhergesagten Wahrscheinlichkeiten. Mit `lr.predict_proba(X_test)[:, 1]` erhältst du die Wahrscheinlichkeit für Klasse 1. Wende deinen Grenzwert darauf an und übergib das Ergebnis an `confusion_matrix`:

```python
proba = lr.predict_proba(X_test)[:, 1]
y_pred = (proba > 0.3).astype(int)
confusion_matrix(y_test, y_pred)
```

Ein niedrigerer Grenzwert von 0,3 sagt mehr Fälle als Klasse 1 voraus. Dadurch steigt häufig der Recall, weil weniger positive Fälle übersehen werden; gleichzeitig können mehr False Positives entstehen.
</details>

# Zusammenfassung: Vom Datensatz zur Vorhersage

- Das Notebook bildet den üblichen Ablauf ab: `X` und `y` festlegen, Daten aufteilen, `LogisticRegression` mit `fit` trainieren und Vorhersagen für `X_test` erstellen.
- Die Konfusionsmatrix zeigt, wie viele tatsächliche Annahmen und Ablehnungen richtig oder falsch vorhergesagt wurden. Accuracy fasst alle richtigen Vorhersagen in einem Anteil zusammen.
- `predict()` verwendet den Standard-Schwellenwert des Modells. Mit `predict_proba()` und einem selbst gewählten Grenzwert wie 0,3 lässt sich die Anzahl der positiven Vorhersagen verändern. Ein niedrigerer Grenzwert erkennt meist mehr angenommene Personen, kann aber auch mehr falsche positive Vorhersagen erzeugen.
- `coef_` und `intercept_` sind gelernte Modellparameter; `get_params()` zeigt vor dem Training festgelegte Einstellungen. Regularisierung begrenzt die Modellgewichte, um übermäßig komplexe Anpassungen zu erschweren.

## Grenzen und praktische Schlussfolgerung
Dieses Beispiel verwendet eine kleine, vorbereitete Kandidatentabelle. Es enthält keine im Notebook dokumentierten Kennzahlen, weil die Code-Zellen nicht ausgeführt wurden; die konkreten Werte sind nach dem Ausführen der Auswertungszellen abzulesen. Für eine belastbare Bewertung sollte man zusätzlich prüfen, ob die Klassen ausgewogen sind und ob das Modell auf weiteren Daten stabil bleibt. Ein einzelner Train-Test-Split ist keine Cross-Validation: Cross-Validation wiederholt die Aufteilung, damit die Bewertung weniger von einem zufälligen Split abhängt.